# Prompt Engineering Study for Finance

Interactive version of the study in [github.com/sarayurkotha/prompt-engineering-study-finance](https://github.com/sarayurkotha/prompt-engineering-study-finance).

Tests 13 prompt techniques on the same task - summarising the key risks from a real JPMorgan Chase 10-K excerpt - and ranks them on accuracy, consistency, and format compliance against a fixed ground-truth checklist.

**Before running:** click the key icon (🔑) in the left sidebar → add a new secret named `GEMINI_API_KEY` → paste in a free key from [aistudio.google.com](https://aistudio.google.com) (no credit card needed) → toggle notebook access on. Your key stays in your own Colab session; it's never written into this notebook file.

In [ ]:
!pip install -q google-genai

!git clone -q https://github.com/sarayurkotha/prompt-engineering-study-finance.git repo
%cd repo/src

import sys
sys.path.insert(0, '.')
from prompts import TECHNIQUES

from google.colab import userdata
from google import genai
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print(f'Loaded {len(TECHNIQUES)} techniques. Client ready.')

## Run the full study

Choose how many runs per technique (the published results use 5) and click Run. The free tier allows roughly 4-5 requests/minute, so this takes about 13 seconds per call - 5 runs x 13 techniques is ~14 minutes.

In [ ]:
n_runs = 5  #@param {type:"slider", min:1, max:5, step:1}

import time, json
from run_study import load_excerpt, call_gemini

excerpt = load_excerpt()
raw_results = []
total = len(TECHNIQUES) * n_runs
done = 0
for technique in TECHNIQUES:
    for run_index in range(n_runs):
        try:
            text, elapsed = call_gemini(client, technique, excerpt)
            error = None
        except Exception as exc:
            text, elapsed, error = '', 0.0, str(exc)
        raw_results.append({'technique_id': technique['technique_id'], 'run_index': run_index, 'raw_text': text, 'elapsed_s': elapsed, 'error': error})
        done += 1
        print(f"[{done}/{total}] {technique['technique_id']} run {run_index+1}/{n_runs}")
        time.sleep(13)  # free tier rate limit - see run_study.py for why

with open('../results/raw_runs.jsonl', 'w') as f:
    for row in raw_results:
        f.write(json.dumps(row) + '\n')
print('Done.')

## Score and view the leaderboard

In [ ]:
%cd ..
!python src/score_and_rank.py

from IPython.display import Image, display
display(Image('outputs/leaderboard.png'))
display(Image('outputs/score_breakdown.png'))

## Try the winning prompt yourself

Paste your own risk-disclosure text (from any company's annual report) and see the top-ranked technique's output live.

In [ ]:
your_text = "Paste a risk disclosure paragraph here..."  #@param {type:"string"}

import pandas as pd
leaderboard = pd.read_csv('results/leaderboard.csv')
winner_id = leaderboard.iloc[0]['technique_id']
winner = [t for t in TECHNIQUES if t['technique_id'] == winner_id][0]
print(f"Using winning technique: {winner['name']}\n")

text, elapsed = call_gemini(client, winner, your_text)
print(text)